# Estratégia de Reversão à Média — Parte 1: Exploração de Dados e Métodos de Rotulagem

Este notebook cobre:
- Carregamento e exploração dos dados históricos do EURGBP H1
- Engenharia de features (médias móveis e skewness)
- Visualização dos 6 métodos de rotulagem de operações

## 1. Configuração do Ambiente

In [ ]:
import sys
import os

# Adiciona o diretório pai ao path para importar as bibliotecas
sys.path.insert(0, os.path.abspath('..'))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline

from labeling_lib import (
    get_labels_filter,
    get_labels_multiple_filters,
    get_labels_filter_bidirectional,
    get_labels_mean_reversion,
    get_labels_mean_reversion_multi,
    get_labels_mean_reversion_v,
)

%matplotlib inline
print('Ambiente configurado com sucesso.')

## 2. Hiperparâmetros

Ajuste os parâmetros abaixo conforme necessário antes de executar o treinamento.

In [ ]:
hyper_params = {
    'symbol': 'EURGBP_H1',
    'markup':       0.00010,   # spread/comissão em pontos
    'stop_loss':    0.02000,   # stop loss em pontos
    'take_profit':  0.00200,   # take profit em pontos
    'periods':      [i for i in range(5, 300, 30)],  # períodos das médias móveis (features principais)
    'periods_meta': [10],                             # período da skewness (feature do meta-modelo)
    'backward':     datetime(2000, 1, 1),  # início do período de treino
    'forward':      datetime(2021, 1, 1),  # início do período de validação fora da amostra
    'n_clusters':   10,          # número de regimes de mercado (K-Means)
    'rolling':      200,         # janela do filtro Savitzky-Golay
}

print(f"Símbolo: {hyper_params['symbol']}")
print(f"Período de treino: até {hyper_params['forward'].date()}")
print(f"Features principais: {len(hyper_params['periods'])} períodos de médias móveis")
print(f"Clusters K-Means: {hyper_params['n_clusters']}")

## 3. Carregamento dos Dados

In [ ]:
def get_prices() -> pd.DataFrame:
    csv_path = os.path.join('..', hyper_params['symbol'] + '.csv')
    p = pd.read_csv(csv_path, sep=r'\s+')
    pFixed = pd.DataFrame(columns=['time', 'close'])
    pFixed['time'] = p['<DATE>'] + ' ' + p['<TIME>']
    pFixed['time'] = pd.to_datetime(pFixed['time'], format='mixed')
    pFixed['close'] = p['<CLOSE>']
    pFixed.set_index('time', inplace=True)
    pFixed.index = pd.to_datetime(pFixed.index, unit='s')
    return pFixed.dropna()

prices = get_prices()
print(f'Total de candles carregados: {len(prices):,}')
print(f'Período: {prices.index[0].date()} até {prices.index[-1].date()}')
prices.head()

### Visualização dos preços históricos

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(prices.index, prices['close'], linewidth=0.6, alpha=0.8, label='Preço de Fechamento')
ax.axvline(x=hyper_params['forward'], color='red', ls='--', lw=1.5, label='Início da validação OOS')
ax.set_title('EURGBP H1 — Série histórica completa')
ax.set_xlabel('Data')
ax.set_ylabel('Preço')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Engenharia de Features

Para cada candle, são calculadas:
- **Features principais**: médias móveis de diferentes períodos (usadas pelo modelo principal)
- **Features do meta-modelo**: skewness (assimetria) da série de preços (usada para identificar o regime de mercado)

In [ ]:
def get_features(data: pd.DataFrame) -> pd.DataFrame:
    pFixed = data.copy()
    pFixedC = data.copy()
    count = 0

    for i in hyper_params['periods']:
        pFixed[str(count)] = pFixedC.rolling(i).mean()
        count += 1

    for i in hyper_params['periods_meta']:
        pFixed[str(count) + 'meta_feature'] = pFixedC.rolling(i).skew()
        count += 1

    return pFixed.dropna()

dataset = get_features(prices)
print(f'Shape do dataset com features: {dataset.shape}')
print(f'Colunas: {list(dataset.columns)}')

## 5. Métodos de Rotulagem

A rotulagem define, para cada ponto histórico, se aquele momento representa um sinal de **compra (0)**, **venda (1)** ou **neutro (2, descartado)**.

O processo é baseado na distância do preço em relação a uma linha de tendência suavizada — quando o preço se desvia significativamente, espera-se que ele retorne à média.

---

### Método 1 — Filtro Savitzky-Golay com Bandas de Quantis

O preço é suavizado com o filtro Savitzky-Golay. Quando o preço se afasta da tendência além do quantil 55% (desvio positivo), sinal de **venda**. Abaixo do quantil 45% (desvio negativo), sinal de **compra**.

In [ ]:
# Visualização do filtro com bandas de quantis
sample = dataset.iloc[-3000:].copy()  # últimos 3000 candles para visualização

rolling = hyper_params['rolling']
smoothed = savgol_filter(sample['close'].values, window_length=rolling, polyorder=3)
diff = sample['close'].values - smoothed
q_low  = np.quantile(diff, 0.20)
q_high = np.quantile(diff, 0.80)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample.index, sample['close'], color='steelblue', linewidth=0.7, alpha=0.8, label='Preço de Fechamento')
ax.plot(sample.index, smoothed, color='orange', linewidth=1.5, label=f'Suavizado (janela={rolling})')
ax.plot(sample.index, smoothed + q_high, color='green', linestyle='--', linewidth=1, label='Quantil Superior (80%)')
ax.plot(sample.index, smoothed + q_low,  color='red',   linestyle='--', linewidth=1, label='Quantil Inferior (20%)')
ax.set_title('Preço e Filtro com Bandas de Quantis')
ax.set_xlabel('Data')
ax.set_ylabel('Preço')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Aplicar rotulagem — Método 1
labeled_m1 = get_labels_filter(
    dataset.copy(),
    rolling=hyper_params['rolling'],
    quantiles=[0.45, 0.55],
    polyorder=3
)
print(f'Método 1 — Total de exemplos rotulados: {len(labeled_m1):,}')
print(labeled_m1['labels'].value_counts().rename({0.0: 'Compra', 1.0: 'Venda'}))

---

### Método 2 — Múltiplos Filtros com Quantis Dinâmicos

Aplica o filtro Savitzky-Golay em múltiplos períodos (50, 100, 200) e usa quantis calculados em janelas deslizantes, detectando sinais de forma hierárquica.

In [ ]:
labeled_m2 = get_labels_multiple_filters(
    dataset.copy(),
    rolling_periods=[50, 100, 200],
    quantiles=[0.45, 0.55],
    window=100,
    polyorder=3
)
print(f'Método 2 — Total de exemplos rotulados: {len(labeled_m2):,}')
print(labeled_m2['labels'].value_counts().rename({0.0: 'Compra', 1.0: 'Venda'}))

---

### Método 3 — Filtro Bidirecional

Usa parâmetros de filtro **diferentes para compra e venda**, levando em conta a assimetria do comportamento do mercado. O sinal de venda usa `rolling1` e o sinal de compra usa `rolling2`.

In [ ]:
labeled_m3 = get_labels_filter_bidirectional(
    dataset.copy(),
    rolling1=50,
    rolling2=200,
    quantiles=[0.45, 0.55],
    polyorder=3
)
print(f'Método 3 — Total de exemplos rotulados: {len(labeled_m3):,}')
print(labeled_m3['labels'].value_counts().rename({0.0: 'Compra', 1.0: 'Venda'}))

---

### Método 4 — Reversão à Média com Restrição de Lucratividade

Além do desvio em relação à média, exige que a operação seja **de fato lucrativa** em um período futuro aleatório entre `min_l` e `max_l` candles. Isso melhora a qualidade dos rótulos ao eliminar falsos sinais.

In [ ]:
# Visualização do método Spline
sample2 = dataset.copy()
x = np.arange(sample2.shape[0])
y = sample2['close'].values
spl = UnivariateSpline(x, y, k=3, s=0.5)
yHat = spl(np.linspace(x.min(), x.max(), x.shape[0]))
diff2 = y - yHat
q2_low  = np.quantile(diff2, 0.20)
q2_high = np.quantile(diff2, 0.80)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample2.index, y, color='steelblue', linewidth=0.5, alpha=0.7, label='Preço de Fechamento')
ax.plot(sample2.index, yHat, color='orange', linewidth=1.5, label='Spline Suavizado (s=0.5)')
ax.plot(sample2.index, yHat + q2_high, color='green', linestyle='--', linewidth=1, label='Quantil Superior (80%)')
ax.plot(sample2.index, yHat + q2_low,  color='red',   linestyle='--', linewidth=1, label='Quantil Inferior (20%)')
ax.set_title('Preço e Spline com Bandas de Quantis')
ax.set_xlabel('Data')
ax.set_ylabel('Preço')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
labeled_m4 = get_labels_mean_reversion(
    dataset.copy(),
    markup=hyper_params['markup'],
    min_l=1,
    max_l=15,
    rolling=0.5,
    quantiles=[0.45, 0.55],
    method='spline',
    shift=0
)
print(f'Método 4 — Total de exemplos rotulados: {len(labeled_m4):,}')
print(labeled_m4['labels'].value_counts().rename({0.0: 'Compra', 1.0: 'Venda'}))

---

### Método 5 — Reversão à Média Multi-Janela

Combina **múltiplas suavizações spline** com diferentes fatores de suavização. O sinal só é gerado quando **todas as janelas** concordam com o desvio em relação à média.

In [ ]:
labeled_m5 = get_labels_mean_reversion_multi(
    dataset.copy(),
    markup=hyper_params['markup'],
    min_l=1,
    max_l=15,
    windows=[0.2, 0.3, 0.5],
    quantiles=[0.45, 0.55]
)
print(f'Método 5 — Total de exemplos rotulados: {len(labeled_m5):,}')
print(labeled_m5['labels'].value_counts().rename({0.0: 'Compra', 1.0: 'Venda'}))

---

### Método 6 — Reversão à Média Ajustada por Volatilidade

Divide os dados em **20 grupos de volatilidade** e calcula bandas de quantis **dinâmicas para cada grupo**. Isso permite que o sistema se adapte automaticamente a diferentes regimes de volatilidade do mercado.

In [ ]:
labeled_m6 = get_labels_mean_reversion_v(
    dataset.copy(),
    markup=hyper_params['markup'],
    min_l=1,
    max_l=15,
    rolling=0.2,
    quantiles=[0.45, 0.55],
    method='spline',
    shift=0,
    volatility_window=100
)
print(f'Método 6 — Total de exemplos rotulados: {len(labeled_m6):,}')
print(labeled_m6['labels'].value_counts().rename({0.0: 'Compra', 1.0: 'Venda'}))

## 6. Comparativo dos Métodos

Resumo do número de exemplos gerados por cada método de rotulagem.

In [ ]:
summary = pd.DataFrame([
    {'Método': 'M1 — Filtro Savitzky-Golay',        'Total': len(labeled_m1), 'Compra': int((labeled_m1['labels']==0).sum()), 'Venda': int((labeled_m1['labels']==1).sum())},
    {'Método': 'M2 — Múltiplos Filtros',             'Total': len(labeled_m2), 'Compra': int((labeled_m2['labels']==0).sum()), 'Venda': int((labeled_m2['labels']==1).sum())},
    {'Método': 'M3 — Filtro Bidirecional',           'Total': len(labeled_m3), 'Compra': int((labeled_m3['labels']==0).sum()), 'Venda': int((labeled_m3['labels']==1).sum())},
    {'Método': 'M4 — Reversão c/ Lucratividade',    'Total': len(labeled_m4), 'Compra': int((labeled_m4['labels']==0).sum()), 'Venda': int((labeled_m4['labels']==1).sum())},
    {'Método': 'M5 — Reversão Multi-Janela',         'Total': len(labeled_m5), 'Compra': int((labeled_m5['labels']==0).sum()), 'Venda': int((labeled_m5['labels']==1).sum())},
    {'Método': 'M6 — Ajustado por Volatilidade',     'Total': len(labeled_m6), 'Compra': int((labeled_m6['labels']==0).sum()), 'Venda': int((labeled_m6['labels']==1).sum())},
])
summary

---

## Próximo Passo

Continue no notebook **`02_Clusterizacao_e_Treinamento.ipynb`** para:
- Aplicar K-Means para identificar regimes de mercado
- Treinar os modelos CatBoost (principal + meta-modelo)
- Avaliar o desempenho de cada combinação cluster/método